In [174]:
import numpy as np
import pandas as pd

In [175]:
NR_LABELS = 12
INITIAL_LR = 0.30
BATCH_SIZE = 256
NR_EPOCHS = 80

In [176]:
# Per batch
def softmax(array):
    shifted = array - np.max(array, axis = 1, keepdims = True)

    exp_array = np.exp(shifted)
    return_array = exp_array / np.sum(exp_array, axis = 1, keepdims = True)

    return return_array

In [177]:
def train_model(train_set, train_labels, W, b, learning_rate, N,  batch_size):

    indices = np.random.permutation(N)

    for start_of_batch in range(0, N, batch_size):
        batch_indices = indices[start_of_batch : start_of_batch + batch_size]
        
        batch_inputs = train_set[batch_indices]
        batch_labels = train_labels[batch_indices]

        prediction = np.dot(batch_inputs, W) + b

        current_batch_size = len(batch_inputs)

        error = softmax(prediction)
        error[np.arange(current_batch_size), batch_labels] -= 1.0

        dW = np.dot(np.transpose(batch_inputs), error) * (1.0 / current_batch_size)
        db = np.sum(error, axis = 0) * (1.0 / current_batch_size)

        W -= learning_rate * dW
        b -= learning_rate * db

In [178]:
train = np.load("/kaggle/input/competitions/nn-2026-fii-homework-1/train.npz")
test = np.load("/kaggle/input/competitions/nn-2026-fii-homework-1/test.npz")

X_train = train["data"]
y_train = train["labels"]
X_test = test["data"]

In [179]:
print("Full train data:", train["data"].shape)
print("Full train labels:", train["labels"].shape)
print("Test data:", test["data"].shape)

Full train data: (60000, 784)
Full train labels: (60000,)
Test data: (15000, 784)


In [180]:
# Normalize data
X_tr = X_train / 255.0
X_te = X_test / 255.0

dataset_mean = np.mean(X_tr)
X_tr -= dataset_mean
X_te -= dataset_mean

In [181]:
# Initialize training data size, sample size, weights matrix and bias vector
N, Dim = train["data"].shape

W = np.zeros((Dim, NR_LABELS))
b = np.zeros(NR_LABELS)

In [182]:
# Train the model
for epoch in range(NR_EPOCHS):
    if epoch < int(0.6 * NR_EPOCHS):
        learning_rate = INITIAL_LR
    elif epoch < int(0.85 * NR_EPOCHS):
        learning_rate = 0.2 * INITIAL_LR
    else:
        learning_rate = 0.03 * INITIAL_LR
    
    train_model(X_tr, y_train, W, b, learning_rate, N, BATCH_SIZE)

In [183]:
# See how many it gets correct on training data
test_scores = np.dot(X_tr, W) + b
Y_test_prediction = np.argmax(test_scores, axis = 1)

count_correct = int(np.sum(Y_test_prediction == y_train))
print(count_correct)

55951


In [184]:
# Submission
test_scores = np.dot(X_te, W) + b
predictions = np.argmax(test_scores, axis = 1)

submission = pd.DataFrame({
    "ID": range(len(predictions)),
    "target": predictions,
})

submission.to_csv("submission.csv", index = False)
print("Created submission.csv")

Created submission.csv
